# Setup
Please check that loans_clean exists as a view before running this code  

This can be done by running create_clean_view.sql in code/SQL_Queries/

## Imports

In [1]:
import sqlite3
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

## Database Connecting

In [2]:
conn = sqlite3.connect('../data/loans.db')
df = pd.read_sql('SELECT * FROM loans_clean', conn)
conn.close()

print(df.shape)
print(df['is_default'].value_counts(normalize=True))

(1303638, 19)
is_default
0    0.799265
1    0.200735
Name: proportion, dtype: float64


## Prep Features

dropping empty rows from specific numeric fields  
encoding categorical features  

training and testing split for model
scaling features due to different magnitudes

In [ ]:
df = df.dropna(subset=['annual_inc', 'dti', 'revol_util'])

for col in ['term', 'grade', 'sub_grade', 'home_ownership', 'purpose', 'addr_state', 'emp_length']:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str)) # type: ignore

features = ['loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'emp_length',
            'home_ownership', 'annual_inc', 'purpose', 'dti', 'delinq_2yrs',
            'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc']

X = df[features]
y = df['is_default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Models

## Baseline Model (Logistic Regression)

In [ ]:
logreg = LogisticRegression(max_iter=1000, class_weight='balanced')
logreg.fit(X_train_scaled, y_train)

preds = logreg.predict(X_test_scaled)
probs = logreg.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))

              precision    recall  f1-score   support

           0       0.80      1.00      0.89    208213
           1       0.31      0.00      0.00     52291

    accuracy                           0.80    260504
   macro avg       0.56      0.50      0.44    260504
weighted avg       0.70      0.80      0.71    260504

ROC-AUC: 0.5004466655211128


C:\Users\corbi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
C:\Users\corbi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
